In [ ]:
# -------------------------------------------------------------------
# WORKFLOW STATE DEFINITIONS
# -------------------------------------------------------------------

class Section(TypedDict):
    title: str
    content: str

class State(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    tested_files: list
    entrypoint: str
    integration_logs: str
    integration_status: str
    final_project: str

# -------------------------------------------------------------------
# UNIT TESTER
# -------------------------------------------------------------------

def tester_call(state: State):
    """Unit tester: run each file individually with retries."""
    tested_files = []
    for file in state["sections"]:
        filename = file["title"]
        code = file["content"]

        _ = edit_file(file_path=filename, content=code, mode="overwrite")

        logs = []
        success = False
        max_attempts = 3

        for attempt in range(max_attempts):
            run_result = run_safe_bash(f"python {filename}")
            logs.append(f"Attempt {attempt+1}:\n{run_result}")

            if "Error" not in run_result and "Traceback" not in run_result:
                success = True
                break

            # Ask LLM for fix (placeholder)
            fix = code  # Replace with LLM integration
            _ = edit_file(file_path=filename, content=fix, mode="overwrite")
            code = fix

        tested_files.append({
            "name": filename,
            "content": code,
            "unit_test_logs": "\n\n".join(logs),
            "unit_status": "Pass" if success else "Fail"
        })

    return {"tested_files": tested_files}

# -------------------------------------------------------------------
# INTEGRATION TESTER
# -------------------------------------------------------------------

def integration_tester(state: State):
    """Integration tester: run the whole project after unit tests."""
    logs = []
    success = False
    max_attempts = 3
    entrypoint = state.get("entrypoint", "main.py")

    for attempt in range(max_attempts):
        run_result = run_safe_bash(f"python {entrypoint}")
        logs.append(f"Integration attempt {attempt+1}:\n{run_result}")

        if "Error" not in run_result and "Traceback" not in run_result:
            success = True
            break

        # Ask LLM for project-wide fix (placeholder)
        for f in state["tested_files"]:
            _ = edit_file(file_path=f["name"], content=f["content"], mode="overwrite")

    return {
        "integration_logs": "\n\n".join(logs),
        "integration_status": "Pass" if success else "Fail",
    }

# -------------------------------------------------------------------
# SYNTHESIZER
# -------------------------------------------------------------------

def synthesizer(state: State):
    """Combine all tested files + integration results."""
    tested_files = state["tested_files"]

    combined = []
    for f in tested_files:
        combined.append(
            f"# File: {f['name']}\n\n{f['content']}\n\n"
            f"# Unit Test Logs:\n{f['unit_test_logs']}\n"
            f"# Unit Status: {f['unit_status']}\n"
        )

    combined.append(
        f"# Integration Test Logs:\n{state.get('integration_logs', '')}\n"
        f"# Integration Status: {state.get('integration_status', 'Unknown')}\n"
    )

    return {"final_project": "\n\n---\n\n".join(combined)} 